Import all required libraries.

In [99]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,confusion_matrix,recall_score,precision_score,roc_auc_score

In [100]:
try:
  data = pd.read_csv("./Telco_Customer_Churn_lyst1769326950438.csv")
  print(f"\nI have successfully loaded Customer churn data")
  print(f"\n Shape of data : {data.shape}")
except Exception as e:
  print(f"Error during loading file")
  exit()


I have successfully loaded Customer churn data

 Shape of data : (7043, 21)


In [101]:
type(data)

pandas.DataFrame

**Data Cleaning**

Step1: Check the missing  values

In [102]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

Clean TotalCharges

In [103]:
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'],errors="coerce")
original_rows = len(data)
data = data.dropna(subset=['TotalCharges'])
print(f"Clean Total charges: Dropped {original_rows-len(data)} rows with NA")

Clean Total charges: Dropped 11 rows with NA


Note: 1-errors="coerce" ,There may some values which can not be converted to numeric.

2-If the column has maximum values null then we can skip that column.

In [104]:
data.info()

<class 'pandas.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   str    
 1   gender            7032 non-null   str    
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   str    
 4   Dependents        7032 non-null   str    
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   str    
 7   MultipleLines     7032 non-null   str    
 8   InternetService   7032 non-null   str    
 9   OnlineSecurity    7032 non-null   str    
 10  OnlineBackup      7032 non-null   str    
 11  DeviceProtection  7032 non-null   str    
 12  TechSupport       7032 non-null   str    
 13  StreamingTV       7032 non-null   str    
 14  StreamingMovies   7032 non-null   str    
 15  Contract          7032 non-null   str    
 16  PaperlessBilling  7032 non-null   str    
 17  PaymentMeth

In [105]:
data.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='str')

Dependant and Target Features

In [106]:
target_feature = 'Churn'
numeric_features = ['tenure','MonthlyCharges','TotalCharges']
categorical_features = [ 'gender', 'SeniorCitizen', 'Partner', 'Dependents','PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport','StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling','PaymentMethod']

Combine All features for X and Y

In [107]:
# X-> Independant features
X= data[numeric_features+categorical_features]
# Y-> Dependant feature or target feature
y= data[target_feature]

Target Churn
0-> Customer Churn
1-> Customer will not churn

Observation: Imbalanced classification problem

In [108]:
from collections import Counter
Counter(y)

Counter({'No': 5163, 'Yes': 1869})

Split the data into train and test data

In [109]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42,stratify=y)
# straitify parameter ensures that class imbalance is represented in both the sets [training and testing]

In [110]:
X_train

,tenure,MonthlyCharges,TotalCharges,gender,SeniorCitizen,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod
4499,12,78.30,909.25,Female,0,No,Yes,Yes,Yes,Fiber optic,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check
1933,20,19.70,415.90,Male,0,No,No,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check
4668,2,61.20,125.95,Female,0,No,Yes,Yes,No,DSL,Yes,No,No,No,No,Yes,Month-to-month,Yes,Credit card (automatic)
5681,34,64.20,2106.30,Female,1,Yes,No,Yes,No,DSL,No,No,Yes,Yes,Yes,No,One year,No,Bank transfer (automatic)
3610,12,100.15,1164.30,Female,0,No,No,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5161,23,54.15,1312.45,Male,0,No,Yes,Yes,No,DSL,No,Yes,No,Yes,No,No,Month-to-month,No,Electronic check
3451,65,70.95,4555.20,Male,1,Yes,No,Yes,No,Fiber optic,No,No,No,No,No,No,One year,Yes,Bank transfer (automatic)
4135,36,92.90,3379.25,Female,0,Yes,Yes,Yes,Yes,DSL,Yes,Yes,Yes,Yes,Yes,Yes,Two year,Yes,Credit card (automatic)
4249,10,65.90,660.05,Female,0,Yes,Yes,Yes,No,DSL,No,Yes,Yes,No,No,Yes,One year,Yes,Mailed check


In [111]:
X_test

,tenure,MonthlyCharges,TotalCharges,gender,SeniorCitizen,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod
4221,1,19.30,19.30,Male,0,No,No,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Credit card (automatic)
1820,6,45.65,323.45,Female,0,Yes,No,Yes,No,DSL,No,No,No,No,No,No,Month-to-month,No,Mailed check
2375,71,109.70,7904.25,Female,1,Yes,No,Yes,No,Fiber optic,Yes,Yes,Yes,Yes,Yes,Yes,Two year,Yes,Bank transfer (automatic)
5462,64,70.15,4480.70,Male,0,Yes,No,Yes,Yes,DSL,No,Yes,No,Yes,No,Yes,One year,Yes,Mailed check
1791,44,61.50,2722.20,Female,0,Yes,No,Yes,No,DSL,Yes,Yes,Yes,No,No,No,One year,Yes,Mailed check
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4685,46,108.90,4854.30,Female,0,Yes,No,Yes,Yes,Fiber optic,No,Yes,Yes,Yes,Yes,Yes,Two year,Yes,Credit card (automatic)
4768,71,19.80,1388.45,Male,1,Yes,Yes,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Credit card (automatic)
6150,32,91.05,2871.50,Male,0,No,No,Yes,No,Fiber optic,No,No,No,No,Yes,Yes,Month-to-month,No,Electronic check
3234,24,19.70,452.55,Female,0,Yes,Yes,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check


In [112]:
y_train = y_train.map({'No':0,'Yes':1})
y_test = y_test.map({'No':0,'Yes':1})
y = y.map({'No':0,'Yes':1})

In [113]:
print(f"Original data churn rate: {y.mean():.4f}")
print(f"Test data churn rate: {y_test.mean():.4f}")
print(f"Train data churn rate: {y_train.mean():.4f}")

Original data churn rate: 0.2658
Test data churn rate: 0.2659
Train data churn rate: 0.2657


Full-Stack Pipeline

In [114]:
from sklearn.compose import ColumnTransformer
# ---1. Numeric Preprocessing Batch
numeric_transformer = Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
    ])
# ---2. Categorical Preprocessing Batch
categorical_transformer= Pipeline(steps=[
    ('imputer',SimpleImputer(strategy="most_frequent")),
     ('onehot',OneHotEncoder(handle_unknown='ignore'))
])
# ---3. Combine the above batches with column transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num',numeric_transformer,numeric_features),
        ('cat',categorical_transformer,categorical_features),
    ],
    remainder="drop"
)

# ---4. Final full stack-pipeline
clf_pipeline = Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('classifier',LogisticRegression(
        class_weight="balanced",
        random_state=42
    ))
])

Model Training on training set

In [115]:
clf_pipeline.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

Model Prediction

In [116]:
y_predict = clf_pipeline.predict(X_test)
print(y_predict)

[0 1 0 ... 1 0 1]


In [117]:
y_prob = clf_pipeline.predict_proba(X_test)[:,1]
print(y_prob)

[0.39717109 0.57209269 0.12870455 ... 0.74641361 0.03938791 0.76153802]


Model Evaluation

In [118]:
print(confusion_matrix(y_test,y_predict))

[[1110  439]
 [ 115  446]]


In [119]:
print(recall_score(y_test,y_predict))

0.7950089126559715


In [120]:
print(precision_score(y_test,y_predict))

0.503954802259887


**Save the model -> Pickle File**

In [121]:
import pickle
pickle.dump(clf_pipeline, open('../model.pkl', 'wb'))

**Load the file and use it for future test data predictions**

In [122]:
model = pickle.load(open('../model.pkl', 'rb'))

In [123]:
y_predict = model.predict(X_test)
print(y_predict)

[0 1 0 ... 1 0 1]
